In [1]:
import pandas as pd
url = '/home/odoo/Documentos/DESARROLLO/Jupyter/Prerrequisitos'



In [2]:
import os
import sys
import pandas as pd
import psycopg2
import cx_Oracle
def query(sql='',option='',parq=[],one=False,dataframe=False,insert=False):
        if option == 'dara_mallas_18':
            conexion = psycopg2.connect("dbname=dara_mallas_18 user=odoo password=root123")
        if option == 'udla_mallas_dev':
            conexion = psycopg2.connect("dbname=udla_mallas user=welintong password=jde.2020")
        if option == 'silabos_dev':
            conexion = psycopg2.connect("dbname=silabos user=welintong password=jde.2020")
        if option == 'dara_mallas':
            conexion = psycopg2.connect("host=10.190.0.168 dbname=dara_mallas user=postgres password=Cia2@22*")
        if option == 'udla_mallas':
            conexion = psycopg2.connect("host=192.168.5.217 dbname=udla_mallas user=wcabascango password=Swealhack99")
        if option == 'banner':
            dsnStr = cx_Oracle.makedsn("snora06.udla-ec.int", "1521", "PRODSTBY")
            conexion = cx_Oracle.connect( user='INT_MALLAS', password='8jAbdj4TWRX8@3q*', dsn=dsnStr)
            #conexion = cx_Oracle.connect("INT_MALLAS/8jAbdj4TWRX8@3q*@//ocora02.udla-ec.int:1521/DBPROD_BA.db.bannerredprod.oraclevcn.com")
        
        if dataframe:
            df_result = pd.read_sql_query(sql=sql, con=conexion, params=parq)
            df_result.columns = map(str.lower, df_result.columns)
            conexion.close()
            return df_result
        else:
            if insert:
                try:
                    cur = conexion.cursor()
                    #cur.execute (sql,parq)
                    cur.execute (sql)
                    conexion.commit()
                    count = cur.rowcount
                    res = str(count)+ " Record inserted successfully into table"
                    if conexion:
                        cur.close()
                        conexion.close()
                        print("PostgreSQL connection is closed")
                    
                    return res
                except Exception as e:
                    print(e)

            else:
                cur = conexion.cursor()
                #cur.execute (sql,parq)
                cur.execute (sql)
                res = [dict((cur.description[i][0], value) \
                        for i, value in enumerate(row)) for row in cur.fetchall()]
                return (res[0] if res else None) if one else res

In [3]:
def get_prerequisite_malla(subject_code,period):
    sql = '''
               					select 
					s.code as subject_code
					,pe.name as period
                    ,p.seq as seq
                    ,p.conector as conector
                    ,p.lparen as lparen
                    ,p.test_code as test_code
                    ,p.test_score as test_score
                    ,ps.code as prerequisite_subject_code
                    ,g.name as grade
                    ,sc.name as score
                    ,rparen
                    ,'homologacion' as prerequsite_type
                    ,s.code as origen
                from  dara_mallas_prerequisite_line pl
                left join dara_mallas_period pe on pl.period_id = pe.id
				left join dara_mallas_prerequisite p on pl.id = p.prerequisite_line_id
                left join dara_mallas_subject ps on p.prerequisite_subject_id = ps.id
				left join dara_mallas_subject s on pl.subject_id = s.id
                left join dara_mallas_grade g on p.grade_id = g.id
                left join dara_mallas_rule_score sc on p.score_id = sc.id
                where (s.code,pe.name) = ( select s.code, max(pe.name)
											from  dara_mallas_prerequisite_line pl
											left join dara_mallas_period pe on pl.period_id = pe.id
											left join dara_mallas_subject s on pl.subject_id = s.id
										  	where  s.code ='%s' and pe.name<='%s'
                                            group by s.code
					)
                    and p.prerequsite_type='malla'
                order by s.code, p.seq
    '''%(subject_code,period)
    res = query(sql=sql,option='dara_mallas_18',dataframe=True)
    return res
def get_prerequisite_all(subject_code,period):
    sql = '''
               					select 
					s.code as subject_code
					,pe.name as period
                    ,p.seq as seq
                    ,p.conector as conector
                    ,p.lparen as lparen
                    ,p.test_code as test_code
                    ,p.test_score as test_score
                    ,ps.code as prerequisite_subject_code
                    ,g.name as grade
                    ,sc.name as score
                    ,rparen
                    ,prerequsite_type
                from  dara_mallas_prerequisite_line pl
                left join dara_mallas_period pe on pl.period_id = pe.id
				left join dara_mallas_prerequisite p on pl.id = p.prerequisite_line_id
                left join dara_mallas_subject ps on p.prerequisite_subject_id = ps.id
				left join dara_mallas_subject s on pl.subject_id = s.id
                left join dara_mallas_grade g on p.grade_id = g.id
                left join dara_mallas_rule_score sc on p.score_id = sc.id
                where (s.code,pe.name) = ( select s.code, max(pe.name)
											from  dara_mallas_prerequisite_line pl
											left join dara_mallas_period pe on pl.period_id = pe.id
											left join dara_mallas_subject s on pl.subject_id = s.id
										  	where  s.code ='%s' and pe.name<='%s' 
                                            group by s.code
					)
                order by s.code, p.seq
    '''%(subject_code,period)
    res = query(sql=sql,option='dara_mallas_18',dataframe=True)
    return res

In [4]:
def get_period_malla_x_subject(subject):
    sql = '''
                select max(pe.name) as period
                from dara_mallas_study_plan_line spl
                left join dara_mallas_study_plan sp on spl.study_plan_id =sp.id
				left join dara_mallas_period pe on sp.period_id = pe.id
                left join dara_mallas_area_homologation ah on spl.area_homologation_id = ah.id
				left join dara_mallas_subject_inherit_area sia ON  ah.id = sia.area_homologation_id
                left join dara_mallas_subject_inherit si on sia.subject_inherit_id = si.id
                left join dara_mallas_subject s on si.subject_id = s.id
                left join dara_mallas_program pr on pr.id=sp.program_id
                --electivas
				  left join dara_mallas_elective_line el on si.elective_id = el.id
				  left join dara_mallas_elective e on el.id = e.elective_line_id
				  left join dara_mallas_subject_inherit siel on e.elective_subject_inherit_id = siel.id
				  left join dara_mallas_subject se on siel.subject_id = se.id
                where (s.code in ('%s') or se.code in ('%s'))
    '''%(subject,subject)
    res = query(sql=sql,option='dara_mallas_18',one=True)
    return res

In [5]:
df_all_prerequisites = pd.DataFrame(columns=['subject_code','period','seq','conector','lparen','test_code','test_score'
            ,'prerequisite_subject_code','grade','score','rparen','prerequsite_type','origen'])
df_homologation_changed = pd.read_csv(url+'/homologation_changed_subjects.csv')
#df_homologation_changed = pd.read_csv(url+'/prerequisite_changed_subjects.csv')
for index , item in df_homologation_changed.iterrows():
    sql = '''
                select s.code as rule 
                , sh.code as homologation_subject_code
                                    , g.name as conector
                                    ,pe.name as period
                from dara_mallas_subject_rule sr
                left join dara_mallas_area a on sr.area_id = a.id
                left join dara_mallas_homologation h on sr.id = h.subject_rule_id
                left join dara_mallas_group_rule g on h.group_id = g.id
                left join dara_mallas_subject s on sr.subject_id = s.id
                left join dara_mallas_subject sh on h.homologation_subject_id = sh.id
                left join dara_mallas_period pe on sr.period_id = pe.id
                where
				s.id <> sh.id
                and s.code = '%s'
                and pe.name ='%s'
            '''%(item['subject_code'],str(item['period']))
    res = query(sql=sql,option='dara_mallas_18',dataframe=True)
    prerequiste_malla = get_prerequisite_malla(item['subject_code'],str(item['period']))
    #if len(prerequiste_malla.values)>0:
    prerequiste_malla['prerequsite_type'] = 'malla'
    df_all_prerequisites= pd.concat([df_all_prerequisites,prerequiste_malla])
    if not res.empty:
        prerequiste_malla['prerequsite_type'] = 'homologacion'
        #siglas que ya escribieron
        write_siglas = []
        #prerequisitos de homologaciones
        for indexh,itemh in res.iterrows():
            #verifica si ya escribio
            if str(itemh['homologation_subject_code'])+str(itemh['period']) not in write_siglas:
                #periodo maximo de la sigla en una malla
                period_max = get_period_malla_x_subject(itemh['homologation_subject_code'])
                datos = []
                prerequisite_homologation = get_prerequisite_all(itemh['homologation_subject_code'],str(itemh['period']))
                prerequisite_homologation['period'] = period_max['period']
                df_all_prerequisites= pd.concat([df_all_prerequisites,prerequisite_homologation])
                if len(prerequisite_homologation.values)>0 and len(prerequiste_malla.values)>0:
                    seq = len(prerequisite_homologation.values)+1
                    #print("aquii",len(prerequisite_homologation.values))
                    datos.append({
                            'subject_code':  itemh['homologation_subject_code']
                            ,'period':   period_max['period']
                            ,'seq':          seq
                            ,'conector':     'O'
                            ,'lparen':       '('
                            ,'test_code':    ''
                            ,'test_score':   ''
                            ,'prerequisite_subject_code':''
                            ,'prerequisite_subject_id':''
                            ,'grade':''
                            ,'grade_id':''
                            ,'score':''
                            ,'score_id':''
                            ,'rparen':''
                            ,'prerequsite_type':'homologacion'
                        ,'origen':''
                        })
                    seq = seq+1
                    
                    for indexi,itemi in prerequiste_malla.iterrows():
                        datos.append({
                            'subject_code':  itemh['homologation_subject_code']
                            ,'period':   period_max['period']
                            ,'seq':          str(seq)
                            ,'conector':     itemi['conector']
                            ,'lparen':       itemi['lparen']
                            ,'test_code':    itemi['test_code']
                            ,'test_score':   itemi['test_score']
                            ,'prerequisite_subject_code':itemi['prerequisite_subject_code']
                            ,'grade':itemi['grade']
                            ,'score':itemi['score']
                            ,'rparen':itemi['rparen']
                            ,'prerequsite_type':itemi['prerequsite_type']
                        ,'origen':itemi['origen']
                        })
                        seq += 1
                    
                    datos.append({
                            'subject_code':  itemh['homologation_subject_code']
                            ,'period':   period_max['period']
                            ,'seq':          seq
                            ,'conector':     ''
                            ,'lparen':       ''
                            ,'test_code':    ''
                            ,'test_score':   ''
                            ,'prerequisite_subject_code':''
                            ,'prerequisite_subject_id':''
                            ,'grade':''
                            ,'grade_id':''
                            ,'score':''
                            ,'score_id':''
                            ,'rparen':')'
                            ,'prerequsite_type':'homologacion'
                        ,'origen':''
                        })
                    seq = seq+1
                    df_all_prerequisites_datos = pd.DataFrame(datos,columns=['subject_code','period','seq','conector','lparen','test_code','test_score'
                    ,'prerequisite_subject_code','grade','score','rparen','prerequsite_type','origen'])
                    df_all_prerequisites= pd.concat([df_all_prerequisites,df_all_prerequisites_datos])
                write_siglas.append(str(itemh['homologation_subject_code'])+str(itemh['period']))

#df_all_prerequisites_finally = pd.DataFrame(df_all_prerequisites)
df_all_prerequisites.to_csv('prerequisitos-PREQ-12062025.csv')  
#df_all_prerequisites.to_csv('or_prerequisites202320-2.csv')  

/tmp/ipykernel_5004/3109504735.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_result = pd.read_sql_query(sql=sql, con=conexion, params=parq)
/tmp/ipykernel_5004/3109504735.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_result = pd.read_sql_query(sql=sql, con=conexion, params=parq)
/tmp/ipykernel_5004/4200252869.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all_prerequisites= pd.concat([df_all_prerequisite